# Introduction to Molecular Dynamics (MD) Simulations

Molecular dynamics simulations are a powerful tool to evaluate a wide range of materials properties and behavior at finite temperatures. Some example applications are to determine thermal expansion, elastic constants, heat transport, analyzing stable equilibrium conditions and many more.

Limitations are mostly based on the sheer computational cost involved with these types of simulations.
- **Limited system sizes**: The maximum size depends on the model used and desired duration of the simulation, common is a few thousand atoms, some examples reach low millions but it is difficult to go beyond that.
- **Short time scales**: Common are time scales of a few ns. Microseconds can be achievable but it is very difficult to capture slow processes. As a result, mathods relying on heating or cooling usually scale the temperature at experimentally not achievable rates. 
- **Classical simulations**: Normal MD simulations rely on classical thermostats and classical equations of motions. Quantum effects especially at low temperatures are neglected making MD simulations only reliable at elevated temperatures. How high depends on the system in question. This issue is more problematic for systems consisting of predominantly light elements such as Hydrogen. There are methods to also account for these effects, e.g. path-integral molecular dynamics.

The primary objective of the MD exercises will be to reproduce the results given in figure 8 in [this paper](https://doi.org/10.1021/acs.jctc.8b01092). We should be able to observe a phase transition of the monoclinic Cu$_2$S cell towards a partially liquid phase which should show substantially different diffusivities.

## Newton's equations of motion

The main objective of performing MD is to solve Newton's equation of motion to predict the propagation of a set of individually moving particles:

$m_i \frac{d^2 \mathbf{r}_i}{dt^2} = \mathbf{F}_i $.

Here, $m_i$ is the mass, $\mathbf{r}_i$ the atomic position, and $\mathbf{F}_i$ the force of particle $i$. There are several approaches that allows us to solve this problem numerically. This typically requires a discretization in time using a finite time step $\Delta t$. Most commonly used in modern MD simulations is the so-called [velocity-Verlet algorithm](https://doi.org/10.1063/1.442716):


$\mathbf{r}_i(t + \Delta t) = \mathbf{r}_i(t) + \left(\mathbf{v}_i(t) + \frac{\Delta t}{2}\mathbf{a}_i(t)\right)\Delta t$,

$\mathbf{v}_i(t + \Delta t) = \mathbf{v}_i(t) + \frac{\Delta t}{2}\left(\mathbf{a}_i(t) + \mathbf{a}_i(t + \Delta t)\right)$.

Here, the acceleration $\mathbf{a}$ is first computed from the forces which depend on the atomistic system and applied model. Subsequently, the new velocities and positions are evaluated using their initial values from the previous time step.

Practical notes:
- Typical values for $\Delta t$ are in the low fs range. Common to use is 1 fs, for very light elements smaller time steps are needed. A lower time step is always better but leads to a proportional increase in computational cost. Ideally, the proper time step should be tested for each specific application.
- MD is commonly conducted via force fields but it can just as well be done using DFT or any other method providing a description for the energies and forces. However, recent advances in MLIPs and their excellent accuracy makes this far less attractive due to its staggering computational cost.
- When choosing an MLIP architecture for MD simulations, try to choose one with conserved and not trained forces as otherwise there will be an energy drift in the system.

## Available implementations for MD

There are many codes that allow the execution of MD simulations. This is not meant to be a comprehensive list, just to give some examples and their major differences.

- **ASE**: The [atomic simulation environment](https://ase-lib.org/ase/md.html) offers easy-to-use tools to conduct MD simulations directly in python. We will be using their implementations for this workshop. As it is rather easy to create an ASE calculator, it can be used with almost any potential and even most DFT codes. Drawbacks are that the implementation is relatively slow and it does not offer a lot of out-of-the-box analysis tools.
- **LAMMPS**: The [Large-scale Atomic/Molecular Massively Parallel Simulator](https://www.lammps.org) is an incredibly powerful and versatile community-driven MD code offering an unparalleled selection of MD manipulation and execution tools. The foundation is a very efficient and parallelizable C++ code, yet support for modern MLIPs is provided via python interfaces. The drawbacks are that it is somewhat challenging to set up and the LAMMPS internal scripting language does take some time to get used to.
- **torch-sim**: [TorchSim](https://github.com/TorchSim/torch-sim) aims to provide an efficient simulator in particular for large-scale models such as MACE. It will also run much faster than ASE. The drawback is that it is rather new and does not yet support as many features as other options.
- **GROMACS**: very popular code for chemical simulations of large molecular structures such as proteins or lipids. Efficient implementation with the drawback of not supporting crystalline structures as well as other options.





## Initializing MD with ASE using MACE

Setting up the calculator is identical to the example for structure relaxation. Afterwards, we have to create a VelocityVerlet object with the starting structure. Here, we will just use the conventional cubic unit cell of $\text{Cu}_2\text{S}$ containing 12 atoms. 

Important note: ASE uses somewhat strange [time units](https://wiki.fysik.dtu.dk/~askhl/ase-doc/ase/units.html). They are in $\mathrm{Å}\sqrt{\mathrm{amu}/\mathrm{eV}}$. It is recommended to use the constants contained in `ase.units` for reliable conversion. Energy and distance units are $\mathrm{eV}$ and $\mathrm{Å}$ as usual.

In [ ]:
from mace.calculators import MACECalculator
import ase.io
import ase.units
from pathlib import Path
import torch

base_path = Path.cwd().parents[1]

# first, we need to choose a model
# we use a cheap model to allow for fast demonstrative simulations
model_file_name = (
    base_path
    / "data"
    / "models"
    / "full_dataset"
    / "M_2_ell_2_16_16_cut_5"
    / "Cu2S_stagetwo.model"
)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# set up the calculator
calc = MACECalculator(model_file_name, device=device)

# load the cubic structure for Cu2S
struct_file_name = base_path / "data" / "Cu2S_FCC" / "Cu2S_conv.extxyz"
struct = ase.io.read(struct_file_name)

# assign the calculator
struct.calc = calc

In [ ]:
from ase.md.verlet import VelocityVerlet

# run the simulation with a time step of 1 fs, we write the log every time step
# into the console output (set by"-")
integrator = VelocityVerlet(
    struct, timestep=1.0 * ase.units.fs, loginterval=1, logfile="-"
)
# run 100 steps
integrator.run(100)


What you will notice above is that nothing happened. The temperature and the kinetic energy are both 0. The reason for this is that we used an optimized starting structure which is in its energetic minimum. You can try to displace one of the atoms by for example changing one of the atom coordinates slightly before the MD execution and run it again. Then, there will be a non-zero force and we will achieve a finite temperature.. For the case of displacing the first atom by `0.01` in x direction, the temperature will only fluctuate between 2 and 5 K. Play around a little bit and try to get the simulation to run closer to room temperature.

Note: since we are doing classical MD simulations, kinetic energy and temperature are innately related via $E_{kin}=\frac{3}{2} N k_B T$ with $N$ being the number of particles.

### The Maxwell-Boltzmann distribution

In practice, one does not try and displace atoms like when starting a pendulum, but rather assign some reasonable initial velocities. One such way of creating initial velocities at a specific temperature is through the Maxwell-Boltzmann distribution. It is given by

$f(\mathbf{v})d^3\mathbf{v}=\left( \frac{m}{2\pi k_B T}\right)^{3/2} e^{-\frac{m \mathbf{v}^2}{2k_B T}}d^3\mathbf{v}$,

where $k_B$ is the Boltzmann constant. The Maxwell-Boltzmann distribution represents the equilibrium situation of an ideal gas at temperature $T$.


In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
import numpy as np

# we restore the starting structure
struct = ase.io.read(struct_file_name)
struct.calc = calc
# for reproducibility it is always good to set a seed or output the seed
# note, that results can be biased if initialization is always based on the
# same seed
rng = np.random.default_rng(seed=27)
# this sets the velocity within the struct at T=300 K
temperature = 300
MaxwellBoltzmannDistribution(struct, temperature_K=temperature, rng=rng)
# this time we will output the trajectory, such that we can view it later
traj_path = (
    base_path / "data" / "MD_trajectories" / f"NVE_MB_{temperature}K.traj"
)
if not traj_path.parent.exists():
    traj_path.parent.mkdir(parents=True)

dt = 1.0
integrator = VelocityVerlet(
    struct,
    timestep=dt * ase.units.fs,
    loginterval=1,
    logfile="-",
    trajectory=traj_path,
)

# run the simulation
n_steps = 500
integrator.run(n_steps)

One thing to note when outputting the trajectories is that they can get very large. We will later discuss how to analyze the results on-the-fly. Usually it is still useful to output the trajectory every ps to troubleshoot possible simulation issues.

## Analyzing the trajectory

The output from MD can be read with a `Trajectory` object. Looping over it will result the individual `ase.atoms.Atoms` objects that will store positions, velocities, energies and forces at each time step. Let's begin by plotting some of these properties.

Let us start by looking at the evolution of temperature over time.

In [ ]:
from ase.io.trajectory import Trajectory
import matplotlib.pyplot as plt

traj = Trajectory(traj_path, "r")


temperatures = []
for atoms in traj:
    temperatures.append(atoms.get_temperature())


plt.plot(np.arange(len(temperatures)) * dt, temperatures)
plt.xlabel("time / fs")
plt.ylabel("temperature / K")
plt.tight_layout()

As you can see, we have pretty large fluctuations. The temperature starts at 300 K but then quickly decays to an equilibrium situation of about 150 K despite our initialization. This is due to us not actually having an ideal gas in our simulation and the reason why for any realistic system a thermostat is needed to heat the system to a stable temperature. It is also the reason why some MD codes do not bother with implementing Maxwell-Boltzmann and just apply an uniform distribution for velocity initialization.

Let us take a look at what actually happened in the simulations by visualizing the trajectory using `ase.visualize.view`.

In [ ]:
from ase.visualize import view

view(traj)

A few windows will pop up, one with the structure shown, one with the title "Movie", one with a plot that by default shows the potential energy of the system and one panel controlling that plot. You can then easily animate the MD run. You will notice that the system performs a finite center-of-mass motion which is a result of the random velocity initialization (it is easier to see if you increase the number of steps). This does not make the simulation incorrect but it might make analysis cumbersome. Luckily, ASE provides a tool for this as well. 

TASK:

Use the `Stationary` function on the `Atoms` object before running the MD and after initializing the velocities to set the center-of-mass motion to zero while preserving the temperature.

```python
from ase.md.velocitydistribution import Stationary
Stationary(struct)
```

Try adding this after initializing the velocities and check whether the center-of-mass motion is still present.

Keep in mind that if you do not reload the atomistic structure above you will effectively "restart" from the last position in the MD run.

For non-periodic systems, there is a similar function called `ZeroRotation` which sets the angular momentum to zero.
